<a href="https://colab.research.google.com/github/doa-2026/machine-learning/blob/main/modelpipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load data

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
path="/content/drive/MyDrive/نسخة من insurance.csv"
df= pd.read_csv(path)
df.head()
df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


Import Packages

In [ ]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline ,Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn import set_config
set_config(transform_output="pandas")
from sklearn.compose import make_column_selector, ColumnTransformer
import numpy as np




In [ ]:
df.duplicated().sum()

np.int64(1)

In [ ]:
df=df.drop_duplicates(keep=False)
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isna().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [ ]:
X=df.drop(columns="charges")
y=df["charges"]


In [ ]:
s=df.select_dtypes(include="object")
for i in s:

  print (df.value_counts(i))
  print("\n")

sex
male      674
female    662
Name: count, dtype: int64


smoker
no     1062
yes     274
Name: count, dtype: int64


region
southeast    364
southwest    325
northeast    324
northwest    323
Name: count, dtype: int64




In [ ]:
z=df.select_dtypes(include="number")
for i in z:
  print (df.value_counts(i))
  print("\n")

age
18    69
19    66
20    29
48    29
47    29
46    29
45    29
52    29
50    29
51    29
25    28
21    28
22    28
23    28
24    28
54    28
53    28
26    28
49    28
28    28
27    28
30    27
29    27
40    27
31    27
41    27
42    27
43    27
44    27
32    26
33    26
57    26
34    26
55    26
56    26
39    25
37    25
38    25
35    25
36    25
58    25
59    25
60    23
61    23
62    23
63    23
64    22
Name: count, dtype: int64


bmi
32.300    13
28.310     9
34.100     8
30.875     8
30.800     8
          ..
45.320     1
45.430     1
45.540     1
45.900     1
46.090     1
Name: count, Length: 548, dtype: int64


children
0    572
1    324
2    240
3    157
4     25
5     18
Name: count, dtype: int64


charges
63770.42801    1
1121.87390     1
1131.50660     1
1135.94070     1
1136.39940     1
              ..
1242.81600     1
1242.26000     1
1241.56500     1
1163.46270     1
1149.39590     1
Name: count, Length: 1336, dtype: int64




Perfomer Validation Split

In [ ]:
X=df.drop(columns="charges")
y=df["charges"]
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42)


Column Transformer

In [ ]:
cat_col=X_train.select_dtypes(include="object").columns
num_col=X_train.select_dtypes(include="number").columns
cat_imp=SimpleImputer(strategy="constant",fill_value="missing")
cat_hot=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
cat_pip=make_pipeline(cat_imp,cat_hot)

cat_tup=("catagori",cat_pip, cat_col)
cat_tup

('catagori',
 Pipeline(steps=[('simpleimputer',
                  SimpleImputer(fill_value='missing', strategy='constant')),
                 ('onehotencoder',
                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
 Index(['sex', 'smoker', 'region'], dtype='object'))

In [ ]:
scale=StandardScaler()
num_imp=SimpleImputer(strategy="mean")
num_pip=make_pipeline(num_imp,scale)
num_tup=("numeric",num_pip,num_col)
num_tup

('numeric',
 Pipeline(steps=[('simpleimputer', SimpleImputer()),
                 ('standardscaler', StandardScaler())]),
 Index(['age', 'bmi', 'children'], dtype='object'))

In [ ]:
col_tra=  ColumnTransformer([cat_tup,num_tup],verbose_feature_names_out=False)
col_tra.fit(X_train)
X_train_preprocessing=col_tra.transform(X_train)
X_test_preprocessing=col_tra.transform(X_test)
X_train_preprocessing.dtypes

,0
sex_female,float64
sex_male,float64
smoker_no,float64
smoker_yes,float64
region_northeast,float64
region_northwest,float64
region_southeast,float64
region_southwest,float64
age,float64
bmi,float64


Create Model Pipline

In [ ]:
lin_reg=LinearRegression()
lin_reg_pip=make_pipeline(col_tra,lin_reg)
lin_reg_pip.fit(X_train,y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('catagori',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['sex', 'smoker', 'region'], dtype='object')),
                                                 ('numeric',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'bmi', 'children'], dtype='object'))],
                                   verbose_feature_names_out=False)),
                ('linearregression', LinearRegression())])

In [ ]:
def regmatric(y_true, y_pred):

 mae = mean_absolute_error(y_true, y_pred)
 mse = mean_squared_error(y_true, y_pred)
 rmse= np.sqrt(mse)
 mre = r2_score(y_true, y_pred)

 print(f"mae {mae:,.2f}")
 print(f"mse {mse:,.2f}")
 print(f"mre {mre:,.2f}")

 metrics={ "MSE":mse ,"mAE":mae,"RMSE":rmse,"R2":mre}
 return metrics



def evalute (reg ,X_train,X_test,y_train,y_test ):

  y_train_pred=reg.predict(X_train)
  y_test_pred=reg.predict(X_test)
  train_metrics=regmatric(y_train,y_train_pred)
  test_metrics=regmatric(y_test,y_test_pred)

  results=pd.DataFrame([train_metrics,test_metrics],index=["train","test"])

  return results.round(3)


evalute(lin_reg_pip,X_train,X_test,y_train,y_test)

mae 4,115.02
mse 35,546,922.81
mre 0.75
mae 4,297.66
mse 39,874,961.87
mre 0.74


,MSE,mAE,RMSE,R2
train,3.554692e+07,4115.019,5962.124,0.752
test,3.987496e+07,4297.663,6314.662,0.745
